# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 dataset using the `mlcroissant` library and the Croissant metadata standard.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Install the required library
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant metadata and dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes as Python attributes
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Version: {md.version}\nIdentifier: {md.identifier}")
print(f"License: {md.license}")

## 2. Data Overview
Let's inspect available record sets, and fields, using their `@id` identifiers.

In [ ]:
# List the available record sets in the dataset, printing their @id and basic info.
record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets listed directly in top-level metadata. Attempting to list from data sources...")

# mlcroissant auto-populates available record_set @id names using the .record_sets attribute
available_record_sets = list(dataset.record_sets)
if available_record_sets:
    print(f"Found record sets:")
    for rs in available_record_sets:
        print(f"  Record Set @id: {rs}")

    # For each, try to print their fields and columns (by @id)
    for rs in available_record_sets:
        recset = dataset.get_record_set(rs)
        print(f"\nFields for Record Set {rs}:")
        if hasattr(recset, 'fields') and recset.fields:
            for f in recset.fields:
                print(f"  Field @id: {getattr(f, '@id', getattr(f, 'id', str(f)))}   name: {getattr(f, 'name', None)}   dataType: {getattr(f, 'dataType', None)}")
        else:
            print("  [No fields listed]")
else:
    print("No record sets found via mlcroissant API.")

## 3. Data Extraction
Load the data from each record set. Use the record set and field `@id`s you listed above.

In [ ]:
# We'll load all available record sets into dataframes
dfs = {}

for rs_id in available_record_sets:
    print(f"Loading records for {rs_id}...")
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"  Loaded {len(df)} rows, columns: {df.columns.tolist()}")
    else:
        print("  [No records loaded]")

# For demonstration, select the first record set loaded (if any):
if dfs:
    example_rs_id = list(dfs)[0]
    print(f"\nExample record set: {example_rs_id}")
    print("Columns:", dfs[example_rs_id].columns.tolist())
    display(dfs[example_rs_id].head())
else:
    print("No data loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and some group analysis to the data, referencing fields by `@id`.

In [ ]:
# Identify a numeric field for analysis. We'll select the first numeric column, if it exists.
example_rs = dfs[example_rs_id]
numeric_fields = example_rs.select_dtypes(include=['number']).columns.tolist()

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric fields found. Picking the first column as fallback.")
    numeric_field_id = example_rs.columns[0]

# Filter records based on a numeric threshold (10 as an example)
threshold = 10
if numeric_field_id in example_rs:
    filtered_df = example_rs[example_rs[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize this numeric field for the filtered records
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (preferably a categorical/text one)
    group_fields = example_rs.select_dtypes(exclude=['number']).columns.tolist()
    group_field_id = None
    for col in group_fields:
        # Pick a field that doesn't have too many unique values
        if example_rs[col].nunique() < len(example_rs) // 2:
            group_field_id = col
            break

    if group_field_id:
        print(f"\nGrouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable categorical grouping field found.")
else:
    print(f"Field {numeric_field_id} not available in example record set.")

## 5. Visualization
Visualize distributions or relationships using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in example_rs:
    plt.figure(figsize=(8, 4))
    sns.histplot(example_rs[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show mean values by group as a bar plot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 clinicopathological colorectal cancer dataset using the `mlcroissant` library.
- Inspected available record sets and fields (by `@id`).
- Loaded and briefly explored the data in pandas DataFrames.
- Applied basic filtering, normalization, and grouping.
- Visualized numeric field distributions.

This workflow can be adapted to other datasets and more advanced analyses using Croissant schema and the `mlcroissant` ecosystem.